# 03 · Bidirectional LSTM

2-layer BiLSTM → LayerNorm on the final forward/backward hidden states → MLP head.

**Protocol (identical for all three models — see `lob/`):**
- Chronological split 70 / 15 / 15 (train / val / test), with purging so that no label window crosses a split boundary.
- Early stopping on **validation** loss; the best-validation weights are restored.
- The per-class probability threshold is chosen on **validation** Sharpe and applied unchanged to test.
- Backtest on non-overlapping 10-second steps, fee 1.5 bps × |Δposition|.

In [ ]:
# ── Setup: works on Colab (clones the repo) and locally (run from notebooks/) ──
import os, sys

REPO_URL = "https://github.com/lshangro-png/mat629-lob-prediction.git"

if "google.colab" in sys.modules:
    if not os.path.exists("mat629-lob-prediction"):
        !git clone -q $REPO_URL
    %cd mat629-lob-prediction
    !pip install -q kagglehub pyarrow
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# SUBSET_ROWS = 100_000 uses only the first ~1.2 days (fast, for development);
# None uses all ~1.03M rows (~12 days). Each setting gets its own feature file.
SUBSET_ROWS = None

# Where the feature table lives. On Colab you can point DATA_DIR at Google
# Drive (after drive.mount) so it survives runtime restarts.
DATA_DIR = "data"
RESULTS_DIR = "results" if SUBSET_ROWS is None else f"results/subset_{SUBSET_ROWS}"
FEATURES_PATH = f"{DATA_DIR}/features{'' if SUBSET_ROWS is None else f'_{SUBSET_ROWS}'}.parquet"

In [ ]:
import torch
from lob.data import FEATURE_SETS, load_dataset
from lob.experiment import run_experiment, save_summary
from lob.features import build_dataset

if not os.path.exists(FEATURES_PATH):
    build_dataset(nrows=SUBSET_ROWS, out_path=FEATURES_PATH)   # downloads via kagglehub
df, meta = load_dataset(FEATURES_PATH)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| rows:", len(df))

## Hyperparameters

Defined in `lob/hparams.py` (shared with `scripts/run_model.py`). Edit `HP` here to experiment.

In [ ]:
import copy
from lob.hparams import HPARAMS

HP = copy.deepcopy(HPARAMS["lstm"])
HP

## Run all feature sets

In [ ]:
rows, models = [], {}
for fs in FEATURE_SETS:
    summary, models[fs], _ = run_experiment(
        df, meta, "lstm", fs, HP["model_kwargs"], HP["train_kwargs"],
        HP["seq_len"], HP["batch_size"], device=DEVICE)
    rows.append(summary)

## Summary

In [ ]:
summary = save_summary(rows, "lstm", RESULTS_DIR)
summary.round(4)